# Milestone 1: Dataset Selection & Data Preprocessing
### Project: **Digital Munshi AI**
**Authors:** Rida Abdul Ghaffar

This notebook guides through the exploratory data analysis and data preprocessing steps required to clean, scale, encode, and balance the Pakistani Legal Case dataset.

## 1. Libraries and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.combine import SMOTETomek

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Data Loading & Understanding

In [ ]:
# Load raw dataset
df = pd.read_csv('pakistan_legal_cases_raw.csv')

print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("--- Column Data Types ---")
df.info()

In [ ]:
# Check the first few records
df.head()

In [ ]:
# Summary statistics for numerical feature
df.describe()

In [ ]:
# Summary statistics for categorical features
df.describe(include=['O'])

In [ ]:
# Check missing values
print("--- Missing Values Count ---")
print(df.isnull().sum())

In [ ]:
# Check duplicate records
num_duplicates = df.duplicated().sum()
print(f"Number of duplicate records found: {num_duplicates}")

In [ ]:
# Class distribution (Target Variable)
class_counts = df['primary_category'].value_counts()
print("--- Class Distribution ---")
print(class_counts)

# Visualizing class distribution
sns.countplot(y='primary_category', data=df, order=class_counts.index, palette='viridis')
plt.title('Initial Class Distribution')
plt.xlabel('Count')
plt.ylabel('Category')
plt.show()

## 3. Data Preprocessing

### A. Removing Duplicates

In [ ]:
df_cleaned = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df_cleaned.shape}")

### B. Handling Missing Values
- Numerical (`severity_score`): Median imputation because of possible outliers skewing the mean.
- Categorical (`client_type`): Mode (most frequent) imputation.

In [ ]:
# Impute numerical severity_score
severity_median = df_cleaned['severity_score'].median()
df_cleaned['severity_score'] = df_cleaned['severity_score'].fillna(severity_median)

# Impute categorical client_type
client_mode = df_cleaned['client_type'].mode()[0]
df_cleaned['client_type'] = df_cleaned['client_type'].fillna(client_mode)

print("Missing values after imputation:")
print(df_cleaned.isnull().sum())

### C. Outlier Detection and Treatment
We use the Interquartile Range (IQR) method to detect outliers in `severity_score` and cap them to 1.5 times the IQR.

In [ ]:
# Boxplot before outlier treatment
sns.boxplot(x=df_cleaned['severity_score'])
plt.title('Severity Score Distribution Before Outlier Treatment')
plt.show()

# IQR Calculation
Q1 = df_cleaned['severity_score'].quantile(0.25)
Q3 = df_cleaned['severity_score'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"IQR: {IQR}, Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")

# Outliers Capping
df_cleaned['severity_score'] = np.clip(df_cleaned['severity_score'], lower_bound, upper_bound)

# Boxplot after outlier treatment
sns.boxplot(x=df_cleaned['severity_score'])
plt.title('Severity Score Distribution After Outlier Treatment')
plt.show()

### D. Feature Encoding
One-Hot Encoding for categorical features (`court_level`, `language`, `client_type`).

In [ ]:
categorical_cols = ['court_level', 'language', 'client_type']
ohe = OneHotEncoder(sparse_output=False, drop='first')
encoded_cats = ohe.fit_transform(df_cleaned[categorical_cols])
encoded_cats_df = pd.DataFrame(encoded_cats, columns=ohe.get_feature_names_out(categorical_cols))
encoded_cats_df.head()

### E. Text Preprocessing & Vectorization (TF-IDF)
Clean the text queries (remove punctuation, lowercasing) and convert them to numerical vectors using TF-IDF.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text) # Remove special characters
    return text

df_cleaned['clean_query'] = df_cleaned['user_query'].apply(clean_text)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=200, stop_words='english')
tfidf_features = tfidf.fit_transform(df_cleaned['clean_query']).toarray()
tfidf_df = pd.DataFrame(tfidf_features, columns=[f"tfidf_{w}" for w in tfidf.get_feature_names_out()])
tfidf_df.head()

### F. Feature Scaling & Assembly
Scale numeric severity score and assemble the final feature matrix $X$ and label vector $y$.

In [ ]:
# Standardize numeric severity_score
scaler = StandardScaler()
scaled_severity = scaler.fit_transform(df_cleaned[['severity_score']])
scaled_severity_df = pd.DataFrame(scaled_severity, columns=['scaled_severity'])

# Assemble final features matrix
X = pd.concat([scaled_severity_df, encoded_cats_df, tfidf_df], axis=1)
y = df_cleaned['primary_category']

print(f"Features matrix X shape: {X.shape}")
print(f"Labels y shape: {y.shape}")

### G. Class Imbalance Resolution using SMOTE-Tomek

In [ ]:
print("Class distribution before SMOTE-Tomek:")
print(y.value_counts())

# Apply SMOTE-Tomek
smote_tomek = SMOTETomek(random_state=42)
X_res, y_res = smote_tomek.fit_resample(X, y)

print("\nClass distribution after SMOTE-Tomek:")
print(y_res.value_counts())

# Visualizing balanced class distribution
sns.countplot(y=y_res, palette='viridis')
plt.title('Balanced Class Distribution')
plt.xlabel('Count')
plt.ylabel('Category')
plt.show()

## 4. Conclusion & Summary
The dataset has been successfully cleaned, transformed, and balanced. The preprocessing pipeline is now complete and ready for Model Training.